# PEFT 库 QLoRA 实战 - ChatGLM3-6B

通常，模型被量化后不会进一步训练用于下游任务，因为由于权重和激活的较低精度，训练可能不稳定。

但是由于PEFT方法只添加额外的可训练参数，这使得我们可以使用PEFT适配器（Adapter）来训练一个量化模型！将量化与PEFT结合起来可以成为在单个GPU上训练大模型的微调策略。

例如，`QLoRA` 是一种将模型量化为4位然后使用LoRA进行训练的方法，使得在单个16GB GPU（本教程以 NVIDIA T4为例）上微调一个具有65B参数的大模型成为可能。

THUDM Hugging Face 主页：https://huggingface.co/THUDM

## 教程说明

本教程使用 QLoRA 论文中介绍的量化技术：`NF4 数据类型`、`双量化` 和 `混合精度计算`，在 `ChatGLM3-6b` 模型上实现了 QLoRA 微调。并展示了完整的 QLoRA 微调流程，具体如下：

- 数据准备
    - 下载数据集
    - 设计 Tokenizer 函数处理样本（map、shuffle、flatten）
    - 自定义批量数据处理类 DataCollatorForChatGLM
- 训练模型
    - 加载 ChatGLM3-6B 量化模型
    - PEFT 量化模型预处理（prepare_model_for_kbit_training）
    - QLoRA 适配器配置（TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING）
    - 微调训练超参数配置（TrainingArguments）
    - 开启训练（trainer.train)
    - 保存QLoRA模型（trainer.model.save_pretrained)
- [模型推理](peft_chatglm_inference.ipynb)
    - 加载 ChatGLM3-6B 基础模型
    - 加载 ChatGLM3-6B QLoRA 模型（PEFT Adapter）
    - 微调前后对比

In [1]:
# 定义全局变量和参数
model_name_or_path = 'THUDM/chatglm3-6b'  # 模型ID或本地路径
train_data_path = 'HasturOfficial/adgen'    # 训练数据路径
eval_data_path = None                     # 验证数据路径，如果没有则设置为None
seed = 8                                 # 随机种子
max_input_length = 512                    # 输入的最大长度
max_output_length = 1536                  # 输出的最大长度
lora_rank = 4                             # LoRA秩
lora_alpha = 32                           # LoRA alpha值
lora_dropout = 0.05                       # LoRA Dropout率
resume_from_checkpoint = None             # 如果从checkpoint恢复训练，指定路径
prompt_text = ''                          # 所有数据前的指令文本
compute_dtype = 'fp32'                    # 计算数据类型（fp32, fp16, bf16）

In [2]:
from pathlib import Path

# 定义模型保存目录
OUTPUT_DIR = Path(
    "/sigan/ai-ml/models/"
    "finetuned/chatglm3-6b-adgen-qlora"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 当前第一轮训练得到的最佳 adapter
BASE_ADAPTER_DIR = Path(
    "/sigan/ai-ml/models/"
    "finetuned/chatglm3-6b-adgen-qlora"
)

# 第二轮训练保存到新目录，避免覆盖第一轮结果
CONTINUED_DIR = Path(
    "/sigan/ai-ml/models/"
    "finetuned/chatglm3-6b-adgen-qlora-continued"
)

CONTINUED_DIR.mkdir(parents=True, exist_ok=True)

## 数据准备

### 下载数据集

从 Hugging Face 加载 adgen 数据集，并tokenize，shuffle

In [3]:
from datasets import load_dataset

dataset = load_dataset(train_data_path)

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['content', 'summary'],
        num_rows: 114599
    })
    validation: Dataset({
        features: ['content', 'summary'],
        num_rows: 1070
    })
})

In [5]:
from datasets import ClassLabel, Sequence
import random
import pandas as pd
from IPython.display import display, HTML

# 数据预览函数，只用于检查数据，不参与训练
def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
        elif isinstance(typ, Sequence) and isinstance(typ.feature, ClassLabel):
            df[column] = df[column].transform(lambda x: [typ.feature.names[i] for i in x])
    display(HTML(df.to_html()))

In [6]:
show_random_elements(dataset["train"], num_examples=3)

,content,summary
0,类型#裙*版型#显瘦*风格#青春*风格#清新*图案#刺绣*裙型#a字*裙长#连衣裙*裙袖长#无袖,这款极具青春魅力和浪漫气息的连衣裙采用精致美观的花朵刺绣设计，清新时尚，穿上去清丽可人又优雅大方。修身大气的长款款式a字裙裙摆设计，完美展现高挑身材，轻松勾勒女性的窈窕身姿，彰显女性的优雅气质。简洁的无袖款式设计，清凉洒脱，让手腕无拘无束。
1,类型#上衣*版型#宽松*风格#街头*风格#简约*风格#休闲*风格#青春*图案#线条*衣样式#卫衣*衣款式#连帽,<UNK><UNK>的卫衣，在演绎街头风的时候，更能轻松应对百变的造型，这款简约款的版式，连时尚混搭风也能轻应对。整体宽松的廓形，显得休闲气质十足，展现出自由从容的姿态，轻松修饰身形线条，不会显胖。动感十足的连帽设计延续经典，契合衣身的造型，平添几分青春肆意的活力。
2,类型#上衣*材质#蕾丝*颜色#白色*风格#简约*风格#休闲*图案#蝴蝶结*图案#蕾丝*衣样式#卫衣*衣袖型#喇叭袖*衣款式#拼接,采用拼接的方式，将白色蕾丝花网拼接于衣身，在胸前和后背环绕出v字型。结合喇叭袖的设计，让造型具有了鲜明的层次感，同时也以精致的花型结构和飘逸质感。为简约休闲的卫衣注入一份唯美浪漫的气息，加上蝴蝶结毛球的装饰，穿上甜美又可爱。


### 使用 ChatGLM3-6b Tokenizer 处理数据


关于 `ignore_label_id` 的设置：

在许多自然语言处理和机器学习框架中，`ignore_label_id` 被设置为 -100 是一种常见的约定。这个特殊的值用于标记在计算损失函数时应该被忽略的目标标签。让我们详细了解一下这个选择的原因：

1. **损失函数忽略特定值**：训练语言模型时，损失函数（例如交叉熵损失）通常只计算对于模型预测重要或关键的标签的损失。在某些情况下，你可能不希望某些标签对损失计算产生影响。例如，在序列到序列的模型中，输入部分的标签通常被设置为一个忽略值，因为只有输出部分的标签对于训练是重要的。

2. **为何选择-100**：这个具体的值是基于实现细节选择的。在 PyTorch 的交叉熵损失函数中，可以指定一个 `ignore_index` 参数。当损失函数看到这个索引值时，它就会忽略对应的输出标签。使用 -100 作为默认值是因为它是一个不太可能出现在标签中的数字（特别是在处理分类问题时，标签通常是从0开始的正整数）。

3. **标准化和通用性**：由于这种做法在多个库和框架中被采纳，-100 作为忽略标签的默认值已经变得相对标准化，这有助于维护代码的通用性和可读性。

总的来说，将 `ignore_label_id` 设置为 -100 是一种在计算损失时排除特定标签影响的便捷方式。这在处理特定类型的自然语言处理任务时非常有用，尤其是在涉及序列生成或修改的任务中。

#### 关于 ChatGLM3 的填充处理说明

- input_id（query）里的填充补全了输入长度，目的是不改变原始文本的含义。
- label（answer）里的填充会用来跟模型基于 query 生成的结果计算 Loss，为了不影响损失值计算，也需要设置。咱们计算损失时，是针对 answer 部分的 Embedding Vector，因此 label 这样填充，前面的序列就自动忽略掉了，只比较生成内容的 loss。因此，需要将answer前面的部分做忽略填充。

In [7]:
from transformers import AutoTokenizer

# revision='b098244' 版本对应的 ChatGLM3-6B 设置 use_reentrant=False
# 最新版本 use_reentrant 被设置为 True，会增加不必要的显存开销
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path,
                                          trust_remote_code=True,
                                          revision='b098244')

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
# tokenize_func 函数
# 把一条原始数据转换成 ChatGLM 能训练的 token 序列
def tokenize_func(example, tokenizer, ignore_label_id=-100):
    """
    对单个数据样本进行tokenize处理。

    参数:
    example (dict): 包含'content'和'summary'键的字典，代表训练数据的一个样本。
    tokenizer (transformers.PreTrainedTokenizer): 用于tokenize文本的tokenizer。
    ignore_label_id (int, optional): 在label中用于填充的忽略ID，默认为-100。

    返回:
    dict: 包含'tokenized_input_ids'和'labels'的字典，用于模型训练。
    """

    # 构建问题文本
    question = prompt_text + example['content']
    if example.get('input', None) and example['input'].strip():
        question += f'\n{example["input"]}'

    # 构建答案文本
    answer = example['summary']

    # 对问题和答案文本进行tokenize处理
    q_ids = tokenizer.encode(text=question, add_special_tokens=False)
    a_ids = tokenizer.encode(text=answer, add_special_tokens=False)

    # 如果tokenize后的长度超过最大长度限制，则进行截断
    if len(q_ids) > max_input_length - 2:  # 保留空间给gmask和bos标记
        q_ids = q_ids[:max_input_length - 2]
    if len(a_ids) > max_output_length - 1:  # 保留空间给eos标记
        a_ids = a_ids[:max_output_length - 1]

    # 构建模型的输入格式
    input_ids = tokenizer.build_inputs_with_special_tokens(q_ids, a_ids)
    question_length = len(q_ids) + 2  # 加上gmask和bos标记

    # 构建标签，对于问题部分的输入使用ignore_label_id进行填充
    labels = [ignore_label_id] * question_length + input_ids[question_length:]

    return {'input_ids': input_ids, 'labels': labels}


In [9]:
column_names = dataset['train'].column_names
tokenized_dataset = dataset['train'].map(
    lambda example: tokenize_func(example, tokenizer),
    batched=False, 
    remove_columns=column_names
)

# 验证集也进行相同的 tokenize
tokenized_valid = dataset["validation"].map(
    lambda example: tokenize_func(
        example,
        tokenizer,
    ),
    batched=False,
    remove_columns=column_names,
)

In [10]:
show_random_elements(tokenized_dataset, num_examples=1)

,input_ids,labels
0,"[64790, 64792, 30910, 33467, 31010, 56778, 30998, 55090, 54888, 31010, 49899, 30998, 56778, 56896, 54625, 31010, 55254, 54584, 56896, 30998, 56778, 40877, 31010, 42875, 30910, 55254, 54584, 56896, 31123, 31628, 57148, 55343, 33355, 34957, 48358, 33403, 31123, 54624, 55715, 56278, 54655, 54807, 54725, 32547, 54603, 33621, 31123, 54532, 51141, 39115, 31123, 54772, 56778, 54715, 53526, 54530, 31921, 31123, 31817, 49899, 54536, 32859, 32144, 31155, 56778, 56164, 52124, 31123, 36546, 46174, 37217, 54530, 42875, 31123, 54883, 56155, 33550, 31155, 2]","[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 30910, 55254, 54584, 56896, 31123, 31628, 57148, 55343, 33355, 34957, 48358, 33403, 31123, 54624, 55715, 56278, 54655, 54807, 54725, 32547, 54603, 33621, 31123, 54532, 51141, 39115, 31123, 54772, 56778, 54715, 53526, 54530, 31921, 31123, 31817, 49899, 54536, 32859, 32144, 31155, 56778, 56164, 52124, 31123, 36546, 46174, 37217, 54530, 42875, 31123, 54883, 56155, 33550, 31155, 2]"


### 数据集处理：shuffle & flatten 

洗牌(shuffle)会将数据集的索引列表打乱，以创建一个索引映射。

然而，一旦您的数据集具有索引映射，速度可能会变慢10倍。这是因为需要额外的步骤来使用索引映射获取要读取的行索引，并且最重要的是，您不再连续地读取数据块。

要恢复速度，需要再次使用 Dataset.flatten_indices()将整个数据集重新写入磁盘上，从而删除索引映射。

ref: https://huggingface.co/docs/datasets/v2.15.0/en/package_reference/main_classes#datasets.Dataset.flatten_indices

In [11]:
tokenized_dataset = tokenized_dataset.shuffle(seed=seed)

In [12]:
tokenized_dataset = tokenized_dataset.flatten_indices()

### 定义 DataCollatorForChatGLM 类 批量处理数据

In [13]:
import torch
from typing import List, Dict, Optional

# DataCollatorForChatGLM 类
# 把一个 batch 中长度不同的样本变成整齐的二维张量
class DataCollatorForChatGLM:
    """
    用于处理批量数据的DataCollator，尤其是在使用 ChatGLM 模型时。

    该类负责将多个数据样本（tokenized input）合并为一个批量，并在必要时进行填充(padding)。

    属性:
    pad_token_id (int): 用于填充(padding)的token ID。
    max_length (int): 单个批量数据的最大长度限制。
    ignore_label_id (int): 在标签中用于填充的ID。
    """

    def __init__(self, pad_token_id: int, max_length: int = 2048, ignore_label_id: int = -100):
        """
        初始化DataCollator。

        参数:
        pad_token_id (int): 用于填充(padding)的token ID。
        max_length (int): 单个批量数据的最大长度限制。
        ignore_label_id (int): 在标签中用于填充的ID，默认为-100。
        """
        self.pad_token_id = pad_token_id
        self.ignore_label_id = ignore_label_id
        self.max_length = max_length

    def __call__(self, batch_data: List[Dict[str, List]]) -> Dict[str, torch.Tensor]:
        """
        处理批量数据。

        参数:
        batch_data (List[Dict[str, List]]): 包含多个样本的字典列表。

        返回:
        Dict[str, torch.Tensor]: 包含处理后的批量数据的字典。
        """
        # 计算批量中每个样本的长度
        len_list = [len(d['input_ids']) for d in batch_data]
        batch_max_len = max(len_list)  # 找到最长的样本长度

        input_ids, labels = [], []
        for len_of_d, d in sorted(zip(len_list, batch_data), key=lambda x: -x[0]):
            pad_len = batch_max_len - len_of_d  # 计算需要填充的长度
            # 添加填充，并确保数据长度不超过最大长度限制
            ids = d['input_ids'] + [self.pad_token_id] * pad_len
            label = d['labels'] + [self.ignore_label_id] * pad_len
            if batch_max_len > self.max_length:
                ids = ids[:self.max_length]
                label = label[:self.max_length]
            input_ids.append(torch.LongTensor(ids))
            labels.append(torch.LongTensor(label))

        # 将处理后的数据堆叠成一个tensor
        input_ids = torch.stack(input_ids)
        labels = torch.stack(labels)

        return {'input_ids': input_ids, 'labels': labels}


In [14]:
# 准备数据整理器
data_collator = DataCollatorForChatGLM(pad_token_id=tokenizer.pad_token_id)

## 训练模型

### 加载 ChatGLM3-6B 量化模型

使用 `nf4` 量化数据类型加载模型，开启双量化配置，以`bf16`混合精度训练，预估显存占用接近4GB

In [15]:
from transformers import AutoModel, BitsAndBytesConfig

_compute_dtype_map = {
    'fp32': torch.float32,
    'fp16': torch.float16,
    'bf16': torch.bfloat16
}

# QLoRA 量化配置
q_config = BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=_compute_dtype_map['bf16'])


### 加载模型


In [16]:
# revision='b098244' 版本对应的 ChatGLM3-6B 设置 use_reentrant=False
# 最新版本 use_reentrant 被设置为 True，会增加不必要的显存开销
model = AutoModel.from_pretrained(model_name_or_path,
                                  quantization_config=q_config,
                                  device_map='auto',
                                  trust_remote_code=True,
                                  revision='b098244')

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [17]:
# 获取当前模型占用的 GPU显存（差值为预留给 PyTorch 的显存）
# 返回模型当前参数和 buffer 大约占用了多少字节。可以近似理解成把模型中所有张量的“元素数量 × 每个元素字节数”加起来
memory_footprint_bytes = model.get_memory_footprint()
memory_footprint_mib = memory_footprint_bytes / (1024 ** 2)  # 转换为 MiB

# 表示 ChatGLM3-6B 使用 NF4 量化加载后，模型权重和相关张量大约占 3.65 GiB
print(f"{memory_footprint_mib:.2f}MiB")

3739.69MiB


### 预处理量化模型

预处理量化后的模型，使其可以支持低精度微调训练

ref: https://huggingface.co/docs/peft/main/en/developer_guides/quantization#quantize-a-model

In [18]:
from peft import TaskType, LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 负责把量化模型调整成“可以进行低精度训练”的状态
# 作用不是增加模型参数，而是先把道路整理好。后面的 get_peft_model() 才会真正加入 LoRA 参数
kbit_model = prepare_model_for_kbit_training(model)

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


### 自定义模型新增 Adapter 

当新的热门 transformer 网络架构（新模型）发布时，Huggingface 社区会尽力快速将它们添加到PEFT中。

如果是 Hugging Face Transformers 库还未内置支持的模型，可以使用自定义模型的方式进行配置。

具体来说，在初始化相应的微调配置类（例如`LoraConfig`）时，我们需要显式指定在哪些层新增适配器（Adapter），并将其设置正确。

ref: https://huggingface.co/docs/peft/developer_guides/custom_models


#### PEFT 适配模块设置


在PEFT库的 [constants.py](https://github.com/huggingface/peft/blob/main/src/peft/utils/constants.py) 文件中定义了不同的 PEFT 方法，在各类大模型上的微调适配模块。

通常，名称相同的模型架构也类似，应用微调方法时的适配器设置也几乎一致。

例如，如果新模型架构是`mistral`模型的变体，并且您想应用 LoRA 微调。在 TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING中`mistral`包含["q_proj", "v_proj"]。

这表示说，对于`mistral`模型，LoRA 的 target_modules 通常是 ["q_proj", "v_proj"]。

- TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING['chatglm']意思是：
  - 对于 ChatGLM，推荐把 LoRA 添加到名字以 query_key_value 结尾的模块上。
  - ChatGLM3 的注意力层没有分别使用 q_proj、k_proj、v_proj，而是用一个融合线性层同时计算 Query、Key、Value，所以目标模块名是：query_key_value

In [19]:
from peft.utils import TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING

target_modules = TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING['chatglm']

In [20]:
target_modules

['query_key_value']

#### LoRA 适配器配置

In [21]:
lora_config = LoraConfig(
    target_modules=target_modules,
    r=lora_rank,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias='none',
    inference_mode=False,
    task_type=TaskType.CAUSAL_LM
)

1. 找到所有 query_key_value 层；
2. 保留原有的量化权重并冻结；
3. 在每层旁边增加两个小的低秩矩阵 A、B；
4. 训练时只更新这些 LoRA 矩阵；
5. 推理时临时把 LoRA 增量加回原权重，或者单独加载 adapter。

In [22]:
qlora_model = get_peft_model(kbit_model, lora_config)

#### 继续训练 LoRA 适配器设置

关键：is_trainable=True
- 默认加载 adapter 是用于推理的，参数会被冻结
- 设置 is_trainable=True 后才会重新允许 LoRA 参数更新

In [23]:
from peft import PeftModel

qlora_model = PeftModel.from_pretrained(
    kbit_model,
    str(BASE_ADAPTER_DIR),
    is_trainable=True,
)

In [24]:
qlora_model.print_trainable_parameters()

trainable params: 974,848 || all params: 6,244,558,848 || trainable%: 0.01561115883009451


### 训练超参数配置

- 1个epoch表示对训练集的所有样本进行一次完整的训练。
- `num_train_epochs` 表示要完整进行多少个 epochs 的训练。

#### 关于使用 num_train_epochs 时，训练总步数 `steps` 的计算方法

- 训练总步数： `total_steps = steps/epoch * num_train_epochs` 
- 每个epoch的训练步数：`steps/epoch = num_train_examples / (batch_size * gradient_accumulation_steps)`


**以 `adgen` 数据集为例计算**

```json
DatasetDict({
    train: Dataset({
        features: ['content', 'summary'],
        num_rows: 114599
    })
    validation: Dataset({
        features: ['content', 'summary'],
        num_rows: 1070
    })
})
```

代入超参数和配置进行计算：

```python
num_train_epochs = 1
num_train_examples = 114599
batch_size = 16
gradient_accumulation_steps = 4


steps = num_train_epochs * num_train_examples / (batch_size * gradient_accumulation_steps)
      = 1 * 114599 / (16 * 4)
      = 1790
```

In [25]:
# from transformers import TrainingArguments, Trainer

# training_args = TrainingArguments(
#     output_dir=f"../models/{model_name_or_path}",          # 输出目录
#     per_device_train_batch_size=16,                     # 每个设备的训练批量大小
#     gradient_accumulation_steps=4,                     # 梯度累积步数
#     # per_device_eval_batch_size=8,                      # 每个设备的评估批量大小
#     learning_rate=1e-3,                                # 学习率
#     num_train_epochs=1,                                # 训练轮数
#     lr_scheduler_type="linear",                        # 学习率调度器类型
#     warmup_ratio=0.1,                                  # 预热比例
#     logging_steps=10,                                 # 日志记录步数
#     save_strategy="steps",                             # 模型保存策略
#     save_steps=100,                                    # 模型保存步数
#     # evaluation_strategy="steps",                       # 评估策略
#     # eval_steps=500,                                    # 评估步数
#     optim="adamw_torch",                               # 优化器类型
#     fp16=True,                                        # 是否使用混合精度训练
# )


In [26]:
# trainer = Trainer(
#         model=qlora_model,
#         args=training_args,
#         train_dataset=tokenized_dataset,
#         data_collator=data_collator
#     )

#### 训练参数（用于演示）

In [27]:
# from transformers import TrainingArguments, Trainer

# training_demo_args = TrainingArguments(
#     output_dir=f"../models/demo/{model_name_or_path}",          # 输出目录
#     per_device_train_batch_size=16,                     # 每个设备的训练批量大小
#     gradient_accumulation_steps=4,                     # 梯度累积步数
#     learning_rate=1e-3,                                # 学习率
#     max_steps=100,                                     # 训练步数
#     lr_scheduler_type="linear",                        # 学习率调度器类型
#     warmup_ratio=0.1,                                  # 预热比例
#     logging_steps=10,                                 # 日志记录步数
#     save_strategy="steps",                             # 模型保存策略
#     save_steps=20,                                    # 模型保存步数
#     optim="adamw_torch",                               # 优化器类型
#     fp16=True,                                        # 是否使用混合精度训练
# )

In [28]:
# trainer = Trainer(
#         model=qlora_model,
#         args=training_demo_args,
#         train_dataset=tokenized_dataset,
#         data_collator=data_collator
#     )

#### 全量参数配置

In [29]:
# from transformers import TrainingArguments, Trainer

# full_args = TrainingArguments(
#     output_dir=str(OUTPUT_DIR),

#     # 当前演示已验证 16×4 能运行
#     per_device_train_batch_size=16,
#     gradient_accumulation_steps=4,
#     per_device_eval_batch_size=8,

#     num_train_epochs=1,
#     learning_rate=1e-3,
#     lr_scheduler_type="linear",
#     warmup_ratio=0.1,

#     logging_steps=10,

#     evaluation_strategy="steps",
#     eval_steps=250,
#     save_strategy="steps",
#     save_steps=250,
#     save_total_limit=8,

#     metric_for_best_model="eval_loss",
#     greater_is_better=False,
#     load_best_model_at_end=True,

#     optim="adamw_torch",    
#     report_to="none",

#     # 统一精度，与 RTX 4080 的能力更匹配
#     bf16=True,
#     fp16=False,
# )

#### 继续训练第二轮参数配置

In [30]:
from transformers import TrainingArguments, Trainer

full_args = TrainingArguments(
    output_dir=str(CONTINUED_DIR),

    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,

    # 这是“额外一轮”，不是总共一轮
    num_train_epochs=1,

    # 已经训练过一轮，继续训练使用更小学习率
    learning_rate=1e-4,
    lr_scheduler_type="linear",
    warmup_ratio=0.03,

    logging_steps=10,

    evaluation_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=8,

    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,

    optim="adamw_torch",
    bf16=True,
    fp16=False,
    report_to="none",
)

In [31]:
trainer = Trainer(
    model=qlora_model,
    args=full_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
)

### 开始训练


In [32]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss,Validation Loss
250,2.996500,3.014331
500,2.965300,3.010170
750,2.966400,3.006669
1000,2.935500,3.004827
1250,2.986300,3.001775
1500,2.974900,2.999328
1750,2.992800,2.997674


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/sigan/miniconda3/envs/peft/lib/p

TrainOutput(global_step=1790, training_loss=2.9840999507371273, metrics={'train_runtime': 16199.6859, 'train_samples_per_second': 7.074, 'train_steps_per_second': 0.11, 'total_flos': 6.918916981222932e+17, 'train_loss': 2.9840999507371273, 'epoch': 1.0})

In [33]:
# 查看最佳 checkpoint
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)

/sigan/ai-ml/models/finetuned/chatglm3-6b-adgen-qlora-continued/checkpoint-1750
2.997673511505127


### 保存模型

In [34]:
# 保存相对路径
# trainer.model.save_pretrained(f"../models/demo/{model_name_or_path}")

# 全量参数训练模型保存
# trainer.save_model(str(OUTPUT_DIR))
# tokenizer.save_pretrained(str(OUTPUT_DIR))

# 继续训练模型保存
trainer.save_model(str(CONTINUED_DIR))
tokenizer.save_pretrained(str(CONTINUED_DIR))
trainer.save_state()

### 加载训练结果并生成文案

In [2]:
import torch

from peft import PeftConfig, PeftModel
from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
)

# 第一轮训练保存路径
ADAPTER_DIR = (
    "/sigan/ai-ml/models/"
    "finetuned/chatglm3-6b-adgen-qlora"
)

# 继续训练保存路径
CONTINUED_DIR = (
    "/sigan/ai-ml/models/"
    "finetuned/chatglm3-6b-adgen-qlora-continued"
)

config = PeftConfig.from_pretrained(CONTINUED_DIR)

In [3]:
BASE_MODEL_REVISION = "b098244"

q_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModel.from_pretrained(
    config.base_model_name_or_path,
    quantization_config=q_config,
    device_map="auto",
    trust_remote_code=True,
    revision=BASE_MODEL_REVISION,
    local_files_only=True,  # 只使用已有缓存，不再联网下载
)

tokenizer = AutoTokenizer.from_pretrained(
    config.base_model_name_or_path,
    trust_remote_code=True,
    revision=BASE_MODEL_REVISION,
    local_files_only=True,
)

base_model.requires_grad_(False)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)
model.eval()

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): ChatGLMForConditionalGeneration(
      (transformer): ChatGLMModel(
        (embedding): Embedding(
          (word_embeddings): Embedding(65024, 4096)
        )
        (rotary_pos_emb): RotaryEmbedding()
        (encoder): GLMTransformer(
          (layers): ModuleList(
            (0-27): 28 x GLMBlock(
              (input_layernorm): RMSNorm()
              (self_attention): SelfAttention(
                (query_key_value): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=4096, out_features=4608, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=4096, out_features=4, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=4, out_features=4608, bias=False)
 

In [4]:
inputs = [
    (
        "类型#裙*版型#显瘦*风格#文艺*风格#简约*"
        "图案#印花*图案#撞色*裙下摆#压褶*"
        "裙长#连衣裙*裙领型#圆领"
    ),
    (
        "类型#裤*版型#显瘦*颜色#纯色*风格#文艺*"
        "风格#简约*风格#休闲*图案#纯色*"
        "裤型#直筒裤*裤口#小脚"
    ),
]

with torch.inference_mode():
    for index, text in enumerate(inputs, start=1):
        response, history = model.chat(
            tokenizer=tokenizer,
            query=text,
            history=[],
        )

        print(f"\n测试 {index}")
        print("输入：", text)
        print("生成广告文案：", response)


测试 1
输入： 类型#裙*版型#显瘦*风格#文艺*风格#简约*图案#印花*图案#撞色*裙下摆#压褶*裙长#连衣裙*裙领型#圆领
生成广告文案： 简约的圆领设计，凸显精致锁骨。修身的版型设计，凸显身材曲线。胸前撞色印花设计，丰富视觉，带来满满的文艺气息。裙摆压褶设计，尽显优雅。

测试 2
输入： 类型#裤*版型#显瘦*颜色#纯色*风格#文艺*风格#简约*风格#休闲*图案#纯色*裤型#直筒裤*裤口#小脚
生成广告文案： 这款裤子采用简约的纯色设计，既休闲又具有时尚气息，同时采用直筒的版型设计，不仅修身显瘦，而且尽显休闲随性气质。小脚的设计，不仅具有收束效果，而且尽显时尚气息。


### 最终结论

建议停止继续训练，并保留两个版本：
- 第一轮：备用版本；
- 第二轮 checkpoint-1750：当前主版本。

最终结论：
1. 第二轮验证 loss 仍在下降，因此不能判定已经过拟合
2. 改善仅约 0.58%，已进入收益递减阶段。生成结果更精炼，当前任务不值得继续追加同数据集训练。
3. 剩余问题适合通过提示词、生成参数和后处理解决。